[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C41_Deep_RL_Course/01_dqn/01_dqn.ipynb)

# 01 · 值函数逼近与 DQN（纯 numpy，从零）

目标：从零手写一个 **numpy DQN**——小 MLP 的 Q 网络（含反向传播）、replay buffer、target network——在 GridWorld toy 环境里学到**显著优于随机**的策略，再加上 **Double**（修过估计）与 **Dueling**（分解 V/A）。

路线：MLP Q 前向+反向 → replay buffer → target network → **完整 DQN 训练** → Double 目标 → Dueling 架构 → ✏️ 练习(Q前向/replay采样/target更新/Double目标) → 📖 答案 → 🧪 真实超参胶囊。

> 心智模型：**Q 网络 = 几个矩阵；TD 目标 = 用冻结副本算的回归靶；DQN = 反复『用稍旧的自己估未来、回归到它、隔段刷新』**。

## 1 · Q 网络：一个小 MLP（前向 + 反向，全手写）

Q 网络输入状态特征（GridWorld 用 one-hot），输出每个动作的 Q 值（长度 `nA`）。
用一个两层 MLP：`h = relu(W1 x + b1)`，`q = W2 h + b2`。

DQN 损失只对**选中动作**那一维回归。我们手写前向、并手算对该维的梯度（链式法则），不依赖任何框架。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

class QNet:
    '''两层 MLP：in_dim -> hidden -> n_actions。手写前向/反向 + Adam。'''
    def __init__(self, in_dim, hidden, n_actions, seed=0):
        r = np.random.default_rng(seed)
        s1 = np.sqrt(2.0 / in_dim); s2 = np.sqrt(2.0 / hidden)
        self.W1 = r.normal(0, s1, (hidden, in_dim)); self.b1 = np.zeros(hidden)
        self.W2 = r.normal(0, s2, (n_actions, hidden)); self.b2 = np.zeros(n_actions)
        # Adam 状态
        self._m = {k: np.zeros_like(v) for k, v in self.params().items()}
        self._v = {k: np.zeros_like(v) for k, v in self.params().items()}
        self._t = 0
    def params(self):
        return {'W1': self.W1, 'b1': self.b1, 'W2': self.W2, 'b2': self.b2}
    def forward(self, X):
        '''X: (B, in_dim) -> Q: (B, n_actions)。缓存中间量供反向。'''
        X = np.atleast_2d(X)
        self._X = X
        self._z1 = X @ self.W1.T + self.b1          # (B,H)
        self._h = np.maximum(0, self._z1)           # relu
        Q = self._h @ self.W2.T + self.b2           # (B,A)
        return Q
    def backward(self, dQ):
        '''dQ: (B, n_actions) = 上游梯度 dL/dQ（已含任何 1/B 归一）。返回参数梯度 dict。
           这里 backward 是纯 Jacobian-向量积，不再内部除以 B（归一化由调用方放进 dQ）。'''
        dW2 = dQ.T @ self._h                         # (A,H)
        db2 = dQ.sum(0)
        dh = dQ @ self.W2                            # (B,H)
        dz1 = dh * (self._z1 > 0)                    # relu 导数
        dW1 = dz1.T @ self._X                        # (H,in)
        db1 = dz1.sum(0)
        return {'W1': dW1, 'b1': db1, 'W2': dW2, 'b2': db2}
    def adam_step(self, grads, lr=1e-3, b1=0.9, b2=0.999, eps=1e-8):
        self._t += 1
        for k, g in grads.items():
            self._m[k] = b1 * self._m[k] + (1 - b1) * g
            self._v[k] = b2 * self._v[k] + (1 - b2) * g * g
            mhat = self._m[k] / (1 - b1 ** self._t)
            vhat = self._v[k] / (1 - b2 ** self._t)
            self.params()[k] -= lr * mhat / (np.sqrt(vhat) + eps)

# 自检：前向输出形状 + 梯度数值检查
qnet = QNet(in_dim=25, hidden=32, n_actions=4, seed=1)
X = rng.standard_normal((8, 25))
Q = qnet.forward(X)
assert Q.shape == (8, 4), 'Q 形状应为 (batch, n_actions)'
print('Q 前向输出形状:', Q.shape)

# 数值梯度检查：对 W2 的一个元素
def loss_fn(net, X, target):
    Q = net.forward(X); return 0.5 * ((Q - target) ** 2).mean(), Q
target = rng.standard_normal((8, 4))
L0, Q = loss_fn(qnet, X, target)
# 损失=0.5*mean((Q-target)^2) over B*A 个元素 -> dL/dQ = (Q-target)/(B*A)
grads = qnet.backward((Q - target) / Q.size)
eps = 1e-5; i, j = 2, 3
qnet.W2[i, j] += eps; Lp, _ = loss_fn(qnet, X, target)
qnet.W2[i, j] -= 2 * eps; Lm, _ = loss_fn(qnet, X, target)
qnet.W2[i, j] += eps
num_grad = (Lp - Lm) / (2 * eps)
print(f'解析梯度={grads["W2"][i,j]:.6f}  数值梯度={num_grad:.6f}')
assert abs(grads['W2'][i, j] - num_grad) < 1e-5, '反向传播应与数值梯度一致'
print('✅ Q 网络前向+反向正确（梯度检查通过）')

## 2 · Replay buffer：打散相关性、复用样本

存 `(s, a, r, s', done)`，随机采小批量。用 `deque(maxlen=N)` 自动丢最旧。
采样要堆成 numpy 数组以便批量前向。**done 必须一起存**——它决定目标是否含未来。

In [ ]:
from collections import deque

class ReplayBuffer:
    def __init__(self, capacity=10000, seed=0):
        self.buf = deque(maxlen=capacity)
        self.rng = np.random.default_rng(seed)
    def push(self, s, a, r, s2, done):
        self.buf.append((np.asarray(s, dtype=np.float32), int(a),
                         float(r), np.asarray(s2, dtype=np.float32), float(done)))
    def __len__(self):
        return len(self.buf)
    def sample(self, batch):
        idx = self.rng.integers(0, len(self.buf), size=batch)
        S = np.stack([self.buf[i][0] for i in idx])
        A = np.array([self.buf[i][1] for i in idx])
        R = np.array([self.buf[i][2] for i in idx], dtype=np.float32)
        S2 = np.stack([self.buf[i][3] for i in idx])
        D = np.array([self.buf[i][4] for i in idx], dtype=np.float32)
        return S, A, R, S2, D

rb = ReplayBuffer(capacity=100, seed=0)
for k in range(150):                       # 推 150 个，超容量 100
    rb.push(np.full(25, k), k % 4, 0.1 * k, np.full(25, k + 1), k % 7 == 0)
assert len(rb) == 100, 'deque 应只保留最近 100 个'
S, A, R, S2, D = rb.sample(16)
assert S.shape == (16, 25) and A.shape == (16,) and D.shape == (16,)
print(f'buffer 大小={len(rb)} (容量100), 采样批形状 S={S.shape}, A={A.shape}, done={D.shape}')
print('✅ replay buffer：FIFO 丢旧 + 随机批采样正确')

## 3 · Target network 与 TD 目标

目标 `y = r + (1-done) * γ * max_a' Q_target(s', a')`。**用冻结的目标网络算**，且 `done` 屏蔽未来。

我们克隆一份 Q 网络作 target，写出 TD 目标计算，并验证 done 的屏蔽逻辑。

In [ ]:
import copy

def clone_net(net):
    return copy.deepcopy(net)

def td_target(target_net, R, S2, D, gamma=0.99):
    '''标准 DQN 目标：r + (1-done)*gamma*max_a' Q_target(s',a')。'''
    Q2 = target_net.forward(S2)               # (B,A)
    max_q2 = Q2.max(axis=1)                    # (B,)
    return R + (1.0 - D) * gamma * max_q2

qnet = QNet(25, 32, 4, seed=2)
tnet = clone_net(qnet)
# 构造一批：一半 done=1（终止），目标应等于 r
S2 = rng.standard_normal((6, 25))
R = np.array([1.0, 0.0, -0.01, 1.0, 0.0, 0.5], dtype=np.float32)
D = np.array([1.0, 0.0, 0.0, 1.0, 0.0, 0.0], dtype=np.float32)
y = td_target(tnet, R, S2, D, gamma=0.99)
print('TD 目标 y =', np.round(y, 3))
# 终止样本(idx 0,3)的目标应严格等于 r（无未来项）
assert abs(y[0] - 1.0) < 1e-6 and abs(y[3] - 1.0) < 1e-6, 'done=1 时 y 应等于 r'
# 非终止样本含未来项，一般 != r
assert not np.allclose(y[1], R[1]), 'done=0 时 y 应含 gamma*maxQ 项'
print('✅ TD 目标正确：done 屏蔽未来，目标用冻结的 target 网络')

## 4 · 完整 DQN：在 GridWorld 上训练

把 Q 网络 + replay + target + ε-贪婪串成训练循环，在 GridWorld 上学。

**这是本模块的高潮**：训练后策略的回报应**显著超过随机基线**（稳健阈值 + 固定 seed 可复现）。

In [ ]:
# GridWorld（与模块00同款，内联以便独立运行）
class GridWorld:
    def __init__(self, n=5, step_cost=0.01):
        self.n=n; self.nS=n*n; self.nA=4; self.step_cost=step_cost; self.goal=n*n-1; self.s=0
    def reset(self): self.s=0; return self.s
    def step(self, a):
        r,c = self.s//self.n, self.s%self.n
        if a==0: r-=1
        elif a==1: r+=1
        elif a==2: c-=1
        elif a==3: c+=1
        r=min(max(r,0),self.n-1); c=min(max(c,0),self.n-1)
        self.s=r*self.n+c; done=(self.s==self.goal)
        return self.s, (1.0 if done else -self.step_cost), done
    def onehot(self, s):
        v=np.zeros(self.nS, dtype=np.float32); v[s]=1.0; return v

def train_dqn(seed=0, episodes=300, gamma=0.95, lr=2e-3, batch=32,
              target_sync=50, eps_start=1.0, eps_min=0.05, eps_decay=0.97,
              double=False):
    np.random.seed(seed)
    env = GridWorld(n=5)
    q = QNet(env.nS, 64, env.nA, seed=seed)
    tnet = clone_net(q)
    rb = ReplayBuffer(capacity=5000, seed=seed)
    erng = np.random.default_rng(seed)
    eps = eps_start; step_count = 0; ep_returns = []
    for ep in range(episodes):
        s = env.reset(); total = 0.0
        for _ in range(100):
            x = env.onehot(s)
            if erng.random() < eps:
                a = int(erng.integers(0, env.nA))
            else:
                a = int(q.forward(x).argmax())
            s2, r, done = env.step(a)
            rb.push(x, a, r, env.onehot(s2), done)
            s = s2; total += r; step_count += 1
            if len(rb) >= batch:
                S, A, R, S2, D = rb.sample(batch)
                if double:
                    a_sel = q.forward(S2).argmax(axis=1)        # 在线网选
                    q2 = tnet.forward(S2)[np.arange(batch), a_sel] # 目标网评
                    y = R + (1 - D) * gamma * q2
                else:
                    y = R + (1 - D) * gamma * tnet.forward(S2).max(axis=1)
                Q = q.forward(S)
                dQ = np.zeros_like(Q)
                pred = Q[np.arange(batch), A]
                # Huber 梯度（delta=1）：clip 到 [-1,1]
                err = np.clip(pred - y, -1.0, 1.0)
                dQ[np.arange(batch), A] = err / batch
                q.adam_step(q.backward(dQ), lr=lr)
                if step_count % target_sync == 0:
                    tnet = clone_net(q)
            if done: break
        eps = max(eps_min, eps * eps_decay)
        ep_returns.append(total)
    return q, ep_returns

q, returns = train_dqn(seed=0, episodes=300)
start = np.mean(returns[:5]); late = np.mean(returns[-30:])
print(f'前5回合平均回报={start:.3f}  后30回合平均回报={late:.3f}  (随机基线≈ -0.5~0)')
# 稳健阈值：后期清晰接近最优(0.9)，且明显高于起步
assert late > 0.8, f'DQN 后期应接近最优(最优≈0.9)，实得 {late:.3f}'
assert late > start + 0.15, 'DQN 后期回报应明显高于起步(学到了)'
print('✅ DQN 学到了！回报爬升到接近最优 0.9（固定 seed 可复现）')

### 看看学到的贪婪策略

把每个状态的 argmax 动作画出来，确认它确实指向目标（右下角）。

In [ ]:
env = GridWorld(n=5)
arrows = ['↑','↓','←','→']
policy_grid = []
for s in range(env.nS):
    if s == env.goal:
        policy_grid.append('G')
    else:
        policy_grid.append(arrows[int(q.forward(env.onehot(s)).argmax())])
print('学到的贪婪策略（应大体指向右下角 G）:')
for r in range(env.n):
    print(' '.join(policy_grid[r*env.n:(r+1)*env.n]))
# 起点(0)的最优首步应是 下 或 右
a0 = int(q.forward(env.onehot(0)).argmax())
assert a0 in (1, 3), '起点最优动作应朝目标(下或右)'
print('✅ 起点策略朝向目标')

## 5 · Double DQN：削过估计（机制对照）

过估计的根源：对**带噪声**的 Q 估计取 `max`，会系统性选中被噪声向上偏的动作，于是 `E[max(Q+noise)] > max(E[Q])`。

我们用一个**受控蒙特卡洛**实验直接量化它：真值 `Q*` 已知、给它加零均值噪声模拟两套独立估计，对比『单估计取 max』(标准 DQN 的目标构造) 与『一套选、另一套评』(Double) 的偏差。这是 Double DQN 的纯机制，干净可复现。

In [ ]:
def overestimation_demo(n_actions=8, noise=1.0, trials=20000, seed=0):
    '''Q* 全相等(=0)，所以 max_a Q* = 0。给 Q* 加噪声后取 max，看偏差。
       标准: 同一套估计既选又评 (max)。Double: 估计A选 argmax，估计B评估其值。'''
    r = np.random.default_rng(seed)
    true_q = np.zeros(n_actions)              # 真值全 0 -> max=0
    single_max, double_val = [], []
    for _ in range(trials):
        qa = true_q + r.normal(0, noise, n_actions)   # 估计 A
        qb = true_q + r.normal(0, noise, n_actions)   # 估计 B(独立)
        single_max.append(qa.max())                   # 标准：A 自选自评
        a_sel = qa.argmax()                           # Double：A 选
        double_val.append(qb[a_sel])                  # B 评
    return float(np.mean(single_max)), float(np.mean(double_val))

bias_single, bias_double = overestimation_demo(n_actions=8, noise=1.0)
print(f'真值 max_a Q* = 0.000')
print(f'标准(自选自评) 的估计 = {bias_single:+.3f}  -> 过估计 {bias_single:+.3f}')
print(f'Double(A选B评) 的估计 = {bias_double:+.3f}  -> 过估计 {bias_double:+.3f}')
assert bias_single > 0.5, '对噪声估计取 max 应明显正偏(过估计)'
assert bias_double < bias_single - 0.3, 'Double(解耦选/评)应显著削弱过估计'
assert abs(bias_double) < 0.2, 'Double 的偏差应接近真值 0'
print('✅ Double DQN 的解耦把过估计从 +0.x 压到≈0 —— 选/评去相关的威力')

### 在真实训练里：Double 也学得到好策略

机制清楚了，再确认带 Double 的完整 DQN **照样学到好策略**（过估计更小、不损性能）。

In [ ]:
q_dbl, returns_dbl = train_dqn(seed=2, episodes=300, double=True)
start_dbl = np.mean(returns_dbl[:5]); late_dbl = np.mean(returns_dbl[-30:])
print(f'Double DQN: 前5={start_dbl:.3f} -> 后30={late_dbl:.3f}')
assert late_dbl > 0.8 and late_dbl > start_dbl + 0.15, 'Double DQN 也应学到接近最优的策略'
print('✅ Double DQN 既削过估计、又不损性能')

## 6 · Dueling 架构：分解 V(s) 与 A(s,a)

`Q(s,a) = V(s) + (A(s,a) - mean_a A(s,a))`。减均值保证可辨识。

我们实现一个 dueling 前向，并验证：**给 V 加常数、A 同时不变，Q 不变**（分解的合理性），以及恒等式 `mean_a(Q-V)=0` 不成立但 `Q` 中 A 部分均值为 0。

In [ ]:
class DuelingQ:
    '''共享主干 -> 两支：V(s) 标量, A(s,a) 向量。Q = V + (A - mean A)。'''
    def __init__(self, in_dim, hidden, n_actions, seed=0):
        r = np.random.default_rng(seed)
        s1 = np.sqrt(2.0/in_dim); s2 = np.sqrt(2.0/hidden)
        self.W1 = r.normal(0,s1,(hidden,in_dim)); self.b1 = np.zeros(hidden)
        self.Wv = r.normal(0,s2,(1,hidden)); self.bv = np.zeros(1)        # 值支
        self.Wa = r.normal(0,s2,(n_actions,hidden)); self.ba = np.zeros(n_actions)  # 优势支
    def forward(self, X):
        X = np.atleast_2d(X)
        h = np.maximum(0, X @ self.W1.T + self.b1)
        V = h @ self.Wv.T + self.bv            # (B,1)
        A = h @ self.Wa.T + self.ba            # (B,Anum)
        Q = V + (A - A.mean(axis=1, keepdims=True))
        return Q, V, A

duel = DuelingQ(25, 32, 4, seed=3)
X = rng.standard_normal((5, 25))
Q, V, A = duel.forward(X)
assert Q.shape == (5, 4) and V.shape == (5, 1) and A.shape == (5, 4)
# 可辨识性：Q 中的优势部分 (A - mean A) 每行均值应为 0
adv_centered = A - A.mean(axis=1, keepdims=True)
assert np.allclose(adv_centered.mean(axis=1), 0.0, atol=1e-10), '中心化优势每行均值=0'
# 因此每行 Q 的均值恰等于 V（优势被中心化掉）
assert np.allclose(Q.mean(axis=1, keepdims=True), V, atol=1e-10), 'mean_a Q(s,a) 应等于 V(s)'
print('Q[0]=', np.round(Q[0],3), ' V[0]=', round(float(V[0,0]),3))
print('✅ Dueling：mean_a Q = V，优势中心化保证可辨识')

---
## ✏️ 练习 1：Q 网络前向 + 选动作

实现 `q_values_and_greedy(net, x)`：对单个状态特征 `x`，返回 `(q_vector, greedy_action)`。
其中 `q_vector` 是长度 `nA` 的一维数组，`greedy_action` 是 argmax 的 int。

In [ ]:
def q_values_and_greedy(net, x):
    # TODO: 用 net.forward(x) 得到 (1, nA)，squeeze 成一维 q_vector；
    #       greedy_action = int(argmax)。返回 (q_vector, greedy_action)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
net = QNet(25, 16, 4, seed=7)
x = np.zeros(25); x[0] = 1.0
qv, ga = q_values_and_greedy(net, x)
assert qv.shape == (4,), 'q_vector 应是一维长度 nA'
assert isinstance(ga, int) and 0 <= ga < 4
assert ga == int(np.argmax(qv)), 'greedy 应是 argmax'
print(f'q_vector={np.round(qv,3)}, greedy={ga}')
print('✅ 练习 1 通过：Q 前向 + 贪婪选动作')

## ✏️ 练习 2：从 replay 采批并算 TD 目标

实现 `compute_targets(target_net, R, S2, D, gamma)`：标准 DQN 目标 `y = r + (1-done)*gamma*max_a' Q_target(s',a')`，返回长度 B 的一维数组。

In [ ]:
def compute_targets(target_net, R, S2, D, gamma=0.95):
    # TODO: 前向 S2 取每行 max；用 (1-D) 屏蔽终止样本的未来项
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
tnet = QNet(25, 16, 4, seed=8)
S2 = rng.standard_normal((5, 25))
R = np.array([1.0, 0.0, 0.0, 0.5, 1.0], dtype=np.float32)
D = np.array([1.0, 0.0, 0.0, 0.0, 1.0], dtype=np.float32)
y = compute_targets(tnet, R, S2, D, gamma=0.95)
assert y.shape == (5,)
assert abs(y[0]-1.0) < 1e-6 and abs(y[4]-1.0) < 1e-6, 'done=1 -> y=r'
ref = R + (1-D)*0.95*tnet.forward(S2).max(axis=1)
assert np.allclose(y, ref), '应等于标准 DQN 目标公式'
print('✅ 练习 2 通过：TD 目标 + done 屏蔽正确')

## ✏️ 练习 3：硬更新 vs 软更新目标网络

实现两种目标网络同步：
(a) `hard_update(target, online)`：把 online 的参数整体拷给 target；
(b) `soft_update(target, online, tau)`：`θ̄ ← τ θ + (1-τ) θ̄`，逐参数。

（直接原地修改 target 的 `params()` 数组。）

In [ ]:
def hard_update(target, online):
    # TODO: 对 online.params() 的每个键，把值拷进 target 对应数组（原地 [:] = ）
    raise NotImplementedError

def soft_update(target, online, tau=0.01):
    # TODO: target_param[:] = tau*online_param + (1-tau)*target_param
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
on = QNet(25, 16, 4, seed=9)
tg = QNet(25, 16, 4, seed=10)            # 不同初始化
assert not np.allclose(on.W1, tg.W1), '初始应不同'
# 软更新一次：应朝 online 移动一点点，但还没到
before = tg.W1.copy()
soft_update(tg, on, tau=0.1)
assert np.allclose(tg.W1, 0.1*on.W1 + 0.9*before), '软更新公式应正确'
# 硬更新：应完全相等
hard_update(tg, on)
assert np.allclose(tg.W1, on.W1) and np.allclose(tg.W2, on.W2), '硬更新应完全拷贝'
print('✅ 练习 3 通过：硬/软更新都正确')

## ✏️ 练习 4：Double DQN 目标

实现 `double_target(online, target, R, S2, D, gamma)`：
`y = r + (1-done)*gamma * Q_target(s', argmax_a' Q_online(s', a'))`。
即**在线网选动作、目标网评估**。

In [ ]:
def double_target(online, target, R, S2, D, gamma=0.95):
    # TODO: a_sel = argmax over online.forward(S2)；
    #       q2 = target.forward(S2) 在 a_sel 处取值；y = R + (1-D)*gamma*q2
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
on = QNet(25, 16, 4, seed=11); tg = QNet(25, 16, 4, seed=12)
S2 = rng.standard_normal((6, 25))
R = np.array([0.,1.,0.,0.,0.,1.], dtype=np.float32)
D = np.array([0.,1.,0.,0.,0.,1.], dtype=np.float32)
y = double_target(on, tg, R, S2, D, gamma=0.95)
assert y.shape == (6,)
# 手工核对
a_sel = on.forward(S2).argmax(axis=1)
q2 = tg.forward(S2)[np.arange(6), a_sel]
assert np.allclose(y, R + (1-D)*0.95*q2), 'Double 目标应用在线网选、目标网评'
# 与标准目标一般不同（除非两网 argmax 一致）
print('✅ 练习 4 通过：Double DQN 目标（解耦选与评）')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1
def q_values_and_greedy(net, x):
    q_vector = net.forward(x).squeeze(0)
    return q_vector, int(np.argmax(q_vector))

In [ ]:
# 练习 2
def compute_targets(target_net, R, S2, D, gamma=0.95):
    max_q2 = target_net.forward(S2).max(axis=1)
    return R + (1.0 - D) * gamma * max_q2

In [ ]:
# 练习 3
def hard_update(target, online):
    for k, v in online.params().items():
        target.params()[k][:] = v

def soft_update(target, online, tau=0.01):
    for k, v in online.params().items():
        target.params()[k][:] = tau * v + (1 - tau) * target.params()[k]

In [ ]:
# 练习 4
def double_target(online, target, R, S2, D, gamma=0.95):
    a_sel = online.forward(S2).argmax(axis=1)
    q2 = target.forward(S2)[np.arange(len(R)), a_sel]
    return R + (1.0 - D) * gamma * q2

---
## 🧪 真实数据胶囊：DQN/Rainbow 的真实超参与规模

下面是 **DQN (Nature 2015) 与 Rainbow (2018)** 在 Atari 上的**真实**超参（公开论文）。用它们体会真实 DQN 的规模——以及为什么我们的 toy 版能在几百回合学会，而 Atari 要上千万帧。

In [ ]:
# DQN (Mnih 2015, Nature) 真实超参（Atari）
DQN_HP = dict(
    replay_capacity=1_000_000,      # 100 万转移
    target_sync=10_000,             # 每 1 万步同步目标网
    batch=32, gamma=0.99,
    lr=2.5e-4, eps_start=1.0, eps_end=0.1, eps_decay_frames=1_000_000,
    total_frames=50_000_000,        # 5000 万帧 (~38 天游戏时间)
)
print('DQN(2015) Atari 真实配置:')
for k, v in DQN_HP.items():
    print(f'  {k:20s} = {v:,}' if isinstance(v, int) else f'  {k:20s} = {v}')

# 我们的 toy 配置对比
print('\n我们的 toy DQN: replay=5000, target_sync=50, ~300 回合(~万级步) 即学会')
ratio = DQN_HP['replay_capacity'] / 5000
print(f'真实 replay 比 toy 大 {ratio:.0f}x —— 规模差距，但算法结构完全相同')
assert DQN_HP['target_sync'] > 50, '真实 target_sync 远大于 toy（状态空间大得多）'
print('✅ 同一套机制(replay+target+ε退火)，只是规模与网络从两层 MLP 换成卷积网')

**🧪 胶囊练习**：实现 `epsilon_at_frame(frame, eps_start, eps_end, decay_frames)`：DQN 的 ε 在前 `decay_frames` 帧从 `eps_start` **线性**退火到 `eps_end`，之后保持 `eps_end`。

In [ ]:
def epsilon_at_frame(frame, eps_start=1.0, eps_end=0.1, decay_frames=1_000_000):
    # TODO: 线性插值，frame>=decay_frames 时返回 eps_end
    raise NotImplementedError

In [ ]:
# 自测
assert abs(epsilon_at_frame(0) - 1.0) < 1e-9, 'frame 0 应为 eps_start'
assert abs(epsilon_at_frame(1_000_000) - 0.1) < 1e-9, '到 decay_frames 应为 eps_end'
assert abs(epsilon_at_frame(5_000_000) - 0.1) < 1e-9, '之后保持 eps_end'
mid = epsilon_at_frame(500_000)
assert abs(mid - 0.55) < 1e-6, '中点应线性插值到 0.55'
print('ε(0)=1.0, ε(50万)=0.55, ε(100万)=0.1 ✅ 胶囊练习通过')

In [ ]:
# 📖 胶囊参考答案
def epsilon_at_frame(frame, eps_start=1.0, eps_end=0.1, decay_frames=1_000_000):
    if frame >= decay_frames:
        return eps_end
    frac = frame / decay_frames
    return eps_start + frac * (eps_end - eps_start)

### 小结
- 表格 Q → **Q 网络**：用 MLP 逼近，把 Q-learning 写成回归 `(Q(s,a) - y)²`，但目标 `y` 自己会动（semi-gradient）。
- **致命三要素**（自举+离策略+逼近）→ 可能发散；DQN 用两件武器驯服它：
  - **经验回放**：打散时间相关性、复用样本（数据维度）；
  - **目标网络**：用冻结副本算目标、隔段同步（目标维度）。
- **Double DQN**：在线网选动作、目标网评估，削 `max` 的过估计（几行代码、高性价比）。
- **Dueling**：`Q=V+(A-mean A)`，分解状态值与优势，动作影响小时学 V 更高效。

你已从零写出能学好策略的 DQN。下一站：**模块 02 · 策略优化 PPO 与 SAC** —— 不再学 Q 再贪婪，而是直接优化策略，并处理连续动作。